In [ ]:
import os
import pydicom
import cv2
import numpy as np
from tqdm import tqdm

INPUT_DIR = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train"
OUTPUT_DIR = "/kaggle/working/images_converted"

os.makedirs(OUTPUT_DIR, exist_ok=True)

files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".dicom")]

for file in tqdm(files[:270]):
    png_path = os.path.join(OUTPUT_DIR, file.replace(".dicom", ".png"))

    if os.path.exists(png_path):
        continue

    dcm = pydicom.dcmread(os.path.join(INPUT_DIR, file))
    img = dcm.pixel_array.astype(np.float32)

    # Safe normalize
    if img.max() != img.min():
        img = (img - img.min()) / (img.max() - img.min())
    else:
        img = np.zeros_like(img)

    img = (img * 255).astype(np.uint8)

    # Resize
    img = cv2.resize(img, (224, 224))

    cv2.imwrite(png_path, img)


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

IMAGE_DIR = "/kaggle/input/datasets/vibhavenu/images-converted/images_converted"
PROCESSED_DIR = "/kaggle/working/images_processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

IMG_SIZE = 224

def crop_black_borders(img):
    thresh = cv2.threshold(img, 5, 255, cv2.THRESH_BINARY)[1]
    coords = cv2.findNonZero(thresh)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        img = img[y:y+h, x:x+w]
    return img
def already_processed(output_path):
    return os.path.exists(output_path)
files = [f for f in os.listdir(IMAGE_DIR)]

for file in tqdm(files[:270]):
    if not file.endswith(".png"):
        continue
    
    input_path = os.path.join(IMAGE_DIR, file)
    output_path = os.path.join(PROCESSED_DIR, file)

    if already_processed(output_path):
        continue
        
    img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
    # Crop borders
    img = crop_black_borders(img)

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)

    # Light denoising
    img = cv2.GaussianBlur(img, (3,3), 0)

    # ensure uniform shape
    h, w = img.shape
    scale = IMG_SIZE / max(h, w)
    img = cv2.resize(img, (int(w*scale), int(h*scale)))

    pad_h = IMG_SIZE - img.shape[0]
    pad_w = IMG_SIZE - img.shape[1]

    img = cv2.copyMakeBorder(
        img,
        pad_h//2, pad_h - pad_h//2,
        pad_w//2, pad_w - pad_w//2,
        cv2.BORDER_CONSTANT,
        value=0
    )

    cv2.imwrite(output_path,img)



In [2]:
import os
import pandas as pd

CSV_PATH = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train.csv"
LABEL_DIR = "/kaggle/working/labels"

os.makedirs(LABEL_DIR, exist_ok=True)

IMG_SIZE = 224   # processed images size

df = pd.read_csv(CSV_PATH)
grouped = df.groupby("image_id")

for image_id, rows in grouped:

    label_path = os.path.join(LABEL_DIR, image_id + ".txt")

    # resume-safe
    if os.path.exists(label_path):
        continue

    with open(label_path, "w") as f:

        for _, row in rows.iterrows():

            if row["class_name"] == "No finding":
                continue

            x_min = row["x_min"]
            y_min = row["y_min"]
            x_max = row["x_max"]
            y_max = row["y_max"]

            # convert to YOLO
            x_center = ((x_min + x_max) / 2) / IMG_SIZE
            y_center = ((y_min + y_max) / 2) / IMG_SIZE
            w = (x_max - x_min) / IMG_SIZE
            h = (y_max - y_min) / IMG_SIZE

            class_id = int(row["class_id"])

            f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")
